In [3]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
from google.cloud import bigquery
client = bigquery.Client(project="project-4b1c9a66-e9fc-4877-a90")

def run_query(sql):
    return client.query(sql).to_dataframe()

## INSPECT DATA

In [12]:
run_query("SELECT * FROM merchant_dispute_analytics.disputes LIMIT 10")

,transaction_id,merchant_id,merchant_name,merchant_segment,merchant_city,transaction_date,channel,customer_segment,amount,dispute_reason,dispute_amount,response_days,merchant_won,chargeback_amount
0,52347,1,Merchant_00001,Digital,Bengaluru,2025-03-16 04:00:00+00:00,Card Present,Premium,2944.62,Goods/Services Returned,2944.62,2,0,2944.62
1,224447,1,Merchant_00001,Digital,Bengaluru,2025-09-02 02:00:00+00:00,Card Present,Mass,4682.89,Duplicate Processing,4682.89,7,1,0.00
2,444338,1,Merchant_00001,Digital,Bengaluru,2025-08-13 06:00:00+00:00,Card Present,Mass,1408.23,Duplicate Processing,1408.23,6,0,1408.23
3,198092,12,Merchant_00012,Services,Bengaluru,2025-11-11 12:00:00+00:00,E-commerce,Mass,536.26,Duplicate Processing,536.26,2,1,0.00
4,461129,12,Merchant_00012,Services,Bengaluru,2025-06-06 17:00:00+00:00,Card Present,Premium,572.11,Fraud,572.11,3,1,0.00
5,118411,20,Merchant_00020,Restaurant,Bengaluru,2025-08-24 02:00:00+00:00,E-commerce,Premium,1132.37,Fraud,1132.37,1,1,0.00
6,79109,36,Merchant_00036,Retail,Bengaluru,2025-03-01 19:00:00+00:00,Mobile,Premium,633.86,Fraud,633.86,3,1,0.00
7,121334,36,Merchant_00036,Retail,Bengaluru,2025-03-04 10:00:00+00:00,Mobile,Mass,1887.36,Fraud,1887.36,2,1,0.00
8,330270,37,Merchant_00037,Retail,Bengaluru,2025-09-13 20:00:00+00:00,E-commerce,Mass,766.79,Duplicate Processing,766.79,12,0,766.79
9,63114,40,Merchant_00040,Digital,Bengaluru,2025-12-15 02:00:00+00:00,E-commerce,Mass,260.67,Goods/Services Returned,260.67,2,1,0.00


## CORE KPIs

In [7]:
kpis = run_query("""
    SELECT
        (SELECT COUNT(*) FROM merchant_dispute_analytics.transactions) AS num_transactions,
        COUNT(*) AS num_disputes,
        SUM(dispute_amount) AS total_disputed_amount,
        SUM(chargeback_amount) AS total_chargeback_amount,
        SAFE_DIVIDE(SUM(chargeback_amount), SUM(dispute_amount)) AS chargeback_rate
    FROM merchant_dispute_analytics.disputes
""")
row = kpis.iloc[0]
print(f"{row['num_disputes']:,} disputes out of {row['num_transactions']:,} transactions "
      f"({100*row['num_disputes']/row['num_transactions']:.2f}% dispute rate). "
      f"${row['total_disputed_amount']:,.0f} disputed, ${row['total_chargeback_amount']:,.0f} charged back."
      f" {100*row['chargeback_rate']:.1f}% of disputed dollars ultimately became chargebacks.")

13,806.0 disputes out of 500,000.0 transactions (2.76% dispute rate). $24,794,578 disputed, $9,202,533 charged back. 37.1% of disputed dollars ultimately became chargebacks.


## DISPUTE RATE

In [9]:
rate_result = run_query("""
    SELECT SAFE_DIVIDE(COUNT(DISTINCT d.transaction_id), COUNT(DISTINCT t.transaction_id)) AS dispute_rate
    FROM merchant_dispute_analytics.transactions t
    LEFT JOIN merchant_dispute_analytics.disputes d USING(transaction_id)
""")
print(f"Overall dispute rate is {100*rate_result.iloc[0]['dispute_rate']:.2f}% of all transactions.")

Overall dispute rate is 2.76% of all transactions.


## MONTHLY TREND

In [14]:
monthly = run_query("""
    SELECT FORMAT_DATE('%Y-%m', transaction_date) AS year_month, COUNT(*) AS dispute_count
    FROM merchant_dispute_analytics.disputes GROUP BY year_month ORDER BY year_month
""")
first, last = monthly.iloc[0], monthly.iloc[-1]
pct_change = 100 * (last["dispute_count"] - first["dispute_count"]) / first["dispute_count"]
print(f"{first['year_month']} had {first['dispute_count']} disputes; {last['year_month']} had {last['dispute_count']}.")
print(f"Dispute volume changed {pct_change:+.1f}% from the first to the last month tracked.")

2025-01 had 1121 disputes; 2025-12 had 1082.
Dispute volume changed -3.5% from the first to the last month tracked.


## MONTH OVER MONTH GROWTH

In [18]:
growth = run_query("""
    WITH monthly AS (
        SELECT DATE_TRUNC(transaction_date, MONTH) AS month, COUNT(*) AS dispute_count
        FROM merchant_dispute_analytics.disputes GROUP BY month
    )
    SELECT month, dispute_count,
           SAFE_DIVIDE(dispute_count - LAG(dispute_count) OVER (ORDER BY month),
                       LAG(dispute_count) OVER (ORDER BY month)) AS mom_growth
    FROM monthly ORDER BY month
""")
worst_month = growth.loc[growth["mom_growth"].idxmax()] if growth["mom_growth"].notna().any() else None
if worst_month is not None:
    print(f"The steepest single-month jump was {100*worst_month['mom_growth']:.1f}%, in {worst_month['month'].strftime('%Y-%m')}.")

The steepest single-month jump was 9.0%, in 2025-03.


## DRIVER ANALYSIS

In [21]:
drivers = run_query("""
    SELECT merchant_segment, channel, dispute_reason, COUNT(*) AS dispute_count,
           SUM(chargeback_amount) AS chargeback_amount
    FROM merchant_dispute_analytics.disputes
    GROUP BY merchant_segment, channel, dispute_reason
    HAVING COUNT(*) >= 30
    ORDER BY chargeback_amount DESC
""")
top = drivers.iloc[0]
print(f"The top loss driver is {top['merchant_segment']} / {top['channel']} / {top['dispute_reason']}, "
      f"at ${top['chargeback_amount']:,.0f} across {top['dispute_count']} disputes.")
print(f"This single combination accounts for {100*top['chargeback_amount']/drivers['chargeback_amount'].sum():.1f}% "
      f"of chargeback dollars across all combinations shown.")

The top loss driver is Retail / E-commerce / Fraud, at $335,094 across 495 disputes.
This single combination accounts for 3.7% of chargeback dollars across all combinations shown.


## MERCHANT LEVEL METRICS

In [24]:
merchant_stats = run_query("""
    SELECT
        t.merchant_id, t.merchant_segment,
        COUNT(DISTINCT t.transaction_id) AS trans_count,
        SUM(t.amount) AS trans_value,
        COUNT(DISTINCT d.transaction_id) AS dispute_count,
        SAFE_DIVIDE(COUNT(DISTINCT d.transaction_id), COUNT(DISTINCT t.transaction_id)) AS dispute_rate,
        SUM(d.dispute_amount) AS total_disputed,
        SUM(d.chargeback_amount) AS total_chargeback,
        AVG(d.response_days) AS avg_response_days,
        AVG(d.merchant_won) AS merchant_win_rate
    FROM merchant_dispute_analytics.transactions t
    LEFT JOIN merchant_dispute_analytics.disputes d ON t.transaction_id = d.transaction_id
    GROUP BY t.merchant_id, t.merchant_segment
    ORDER BY total_chargeback DESC
    LIMIT 20
""")
worst_merchant = merchant_stats.iloc[0]
print(f"The highest-chargeback merchant is {worst_merchant['merchant_id']} "
      f"({worst_merchant['merchant_segment']}), with ${worst_merchant['total_chargeback']:,.0f} charged back "
      f"across {worst_merchant['dispute_count']} disputes.")
print(f"This merchant's dispute rate is {100*worst_merchant['dispute_rate']:.2f}%, "
      f"versus a portfolio dispute rate of {100*merchant_stats['dispute_count'].sum()/merchant_stats['trans_count'].sum():.2f}%.")

The highest-chargeback merchant is 1747 (Digital), with $19,408 charged back across 7 disputes.
This merchant's dispute rate is 7.22%, versus a portfolio dispute rate of 5.00%.
